In [5]:
import pandas as pandas
import sys
import yfinance as yf
import datetime
import numpy as np
import matplotlib.pyplot as plt

In [6]:
def get_ticker_data(ticker, start, end):
    t = yf.Ticker(ticker)
    df = t.history(start=start, end=end)
    df.index = df.index.tz_localize(None)
    df['Ticker'] = ticker
        
        # Price features
    df['Daily_Return'] = df['Close'].pct_change()
    df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))
        
        # Moving averages
    df['MA_20'] = df['Close'].rolling(20).mean()
    df['MA_50'] = df['Close'].rolling(50).mean()
    df['MA_200'] = df['Close'].rolling(200).mean()
        
        # Volatility
    df['Volatility_20'] = df['Daily_Return'].rolling(20).std()
        
        # Volume
    df['Volume_Change'] = df['Volume'].pct_change()
    df['Volume_MA_20'] = df['Volume'].rolling(20).mean()


    def compute_rsi(series, period=14):
        delta = series.diff()
        gain = delta.clip(lower=0).rolling(period).mean()
        loss = -delta.clip(upper=0).rolling(period).mean()
        rs = gain / loss
        return 100 - (100 / (1 + rs))
        
        # Momentum
    df['RSI'] = compute_rsi(df['Close'])


    def compute_macd(series, fast=12, slow=26, signal=9):
        ema_fast = series.ewm(span=fast).mean()
        ema_slow = series.ewm(span=slow).mean()
        macd = ema_fast - ema_slow
        signal_line = macd.ewm(span=signal).mean()
        return macd, signal_line

    df['MACD'], df['MACD_Signal'] = compute_macd(df['Close'])
        
        # Bollinger Bands
    df['BB_Upper'] = df['MA_20'] + 2 * df['Close'].rolling(20).std()
    df['BB_Lower'] = df['MA_20'] - 2 * df['Close'].rolling(20).std()
    df['BB_Width'] = df['BB_Upper'] - df['BB_Lower']
    return df

In [7]:
df = get_ticker_data('MSFT', '2010-01-01', '2024-01-01')
print(df.head())

                 Open       High        Low      Close    Volume  Dividends  \
Date                                                                          
2010-01-04  22.831328  23.189232  22.808958  23.077387  38409100        0.0   
2010-01-05  23.002816  23.189225  22.846232  23.084835  49749600        0.0   
2010-01-06  23.025195  23.174323  22.756768  22.943176  58182400        0.0   
2010-01-07  22.838772  22.890968  22.510694  22.704559  50559700        0.0   
2010-01-08  22.577812  23.025192  22.547986  22.861153  51197400        0.0   

            Stock Splits Ticker  Daily_Return  Log_Return  ...  MA_200  \
Date                                                       ...           
2010-01-04           0.0   MSFT           NaN         NaN  ...     NaN   
2010-01-05           0.0   MSFT      0.000323    0.000323  ...     NaN   
2010-01-06           0.0   MSFT     -0.006136   -0.006155  ...     NaN   
2010-01-07           0.0   MSFT     -0.010400   -0.010455  ...     NaN   
20

In [8]:
df = df.drop(columns=['Dividends', 'Stock Splits'], errors='ignore')

In [9]:
df.dropna(inplace=True)

In [10]:
print(df.head())

                 Open       High        Low      Close    Volume Ticker  \
Date                                                                      
2010-10-18  19.359230  19.631576  19.253318  19.533228  48330500   MSFT   
2010-10-19  19.117143  19.192794  18.875058  18.988535  66150900   MSFT   
2010-10-20  19.109588  19.215500  18.988546  19.147413  56283600   MSFT   
2010-10-21  19.215494  19.321408  18.950714  19.230625  50032400   MSFT   
2010-10-22  19.306274  19.321405  19.117146  19.200361  25837900   MSFT   

            Daily_Return  Log_Return      MA_20      MA_50     MA_200  \
Date                                                                    
2010-10-18      0.010963    0.010903  18.719599  18.574419  20.395262   
2010-10-19     -0.027885   -0.028282  18.717707  18.568759  20.374818   
2010-10-20      0.008367    0.008332  18.744185  18.574403  20.355131   
2010-10-21      0.004346    0.004336  18.781633  18.584871  20.336568   
2010-10-22     -0.001574   -0.001575

In [11]:
def get_market_sentiment(start="2020-01-01", end="2024-01-01"):
    sentiment = pandas.DataFrame()
    
    # VIX - Fear index
    vix = yf.download("^VIX", start=start, end=end)['Close'].squeeze()
    vix.index = vix.index.tz_localize(None)  # strip timezone
    sentiment['VIX'] = vix
    sentiment['VIX_MA20'] = vix.rolling(20).mean()
    
    # S&P 500 - Market trend
    sp500 = yf.download("^GSPC", start=start, end=end)['Close'].squeeze()
    sp500.index = sp500.index.tz_localize(None)  # strip timezone
    sentiment['SP500'] = sp500
    sentiment['SP500_Return'] = sp500.pct_change()
    sentiment['SP500_MA50'] = sp500.rolling(50).mean()
    
    # Treasury yield
    treasury = yf.download("^TNX", start=start, end=end)['Close'].squeeze()
    treasury.index = treasury.index.tz_localize(None)
    sentiment['Treasury_10Y'] = treasury
    sentiment['Yield_Change'] = treasury.pct_change()
    
    # Gold
    gold = yf.download("GC=F", start=start, end=end)['Close'].squeeze()
    gold.index = gold.index.tz_localize(None)
    sentiment['Gold_Price'] = gold
    sentiment['Gold_Return'] = gold.pct_change()
    
    # Dollar index
    dxy = yf.download("DX-Y.NYB", start=start, end=end)['Close'].squeeze()
    dxy.index = dxy.index.tz_localize(None)
    sentiment['DXY'] = dxy
    sentiment['DXY_Return'] = dxy.pct_change()

    # Market breadth - fix alignment issue
    sp500_aligned = sentiment['SP500']
    ma50_aligned = sentiment['SP500_MA50']
    sentiment['Market_Regime'] = np.where(
        sp500_aligned > ma50_aligned, 1, 0
    )
    
    return sentiment

sentiment_df = get_market_sentiment()
print(sentiment_df.head())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

              VIX  VIX_MA20        SP500  SP500_Return  SP500_MA50  \
Date                                                                 
2020-01-02  12.47       NaN  3257.850098           NaN         NaN   
2020-01-03  14.02       NaN  3234.850098     -0.007060         NaN   
2020-01-06  13.85       NaN  3246.280029      0.003533         NaN   
2020-01-07  13.79       NaN  3237.179932     -0.002803         NaN   
2020-01-08  13.45       NaN  3253.050049      0.004902         NaN   

            Treasury_10Y  Yield_Change   Gold_Price  Gold_Return        DXY  \
Date                                                                          
2020-01-02         1.882           NaN  1524.500000          NaN  96.849998   
2020-01-03         1.788     -0.049947  1549.199951     0.016202  96.839996   
2020-01-06         1.811      0.012864  1566.199951     0.010973  96.669998   
2020-01-07         1.827      0.008835  1571.800049     0.003576  96.980003   
2020-01-08         1.874      0.025

In [12]:
sentiment_df.dropna(inplace=True)
print(sentiment_df.head())

                  VIX  VIX_MA20        SP500  SP500_Return   SP500_MA50  \
Date                                                                      
2020-03-13  57.830002   35.3995  2711.020020      0.092871  3197.861792   
2020-03-16  82.690002   38.8500  2386.129883     -0.119841  3180.427388   
2020-03-17  75.910004   41.9040  2529.189941      0.059955  3166.314185   
2020-03-18  76.449997   45.0075  2398.100098     -0.051831  3149.350586   
2020-03-19  72.000000   47.8295  2409.389893      0.004708  3132.794785   

            Treasury_10Y  Yield_Change   Gold_Price  Gold_Return         DXY  \
Date                                                                           
2020-03-13         0.951      0.120141  1515.699951    -0.046310   98.750000   
2020-03-16         0.728     -0.234490  1485.900024    -0.019661   98.089996   
2020-03-17         0.997      0.369505  1524.900024     0.026247   99.580002   
2020-03-18         1.266      0.269809  1477.300049    -0.031215  101.1600

In [13]:
def get_full_historical_fundamentals(ticker):
    t = yf.Ticker(ticker)
    
    income = t.quarterly_income_stmt.T
    balance = t.quarterly_balance_sheet.T
    cashflow = t.quarterly_cashflow.T
    
    # Key metrics only
    fundamentals = pandas.concat([income, balance, cashflow], axis=1)
    fundamentals.index = pandas.to_datetime(fundamentals.index)
    fundamentals['Ticker'] = ticker
    
    return fundamentals.sort_index()
fundamentals_df = get_full_historical_fundamentals("MSFT")
print(fundamentals_df.head())

            Tax Effect Of Unusual Items  Tax Rate For Calcs  \
2024-09-30                          NaN                 NaN   
2024-12-31                -2.032200e+08            0.180000   
2025-03-31                 6.966000e+07            0.180000   
2025-06-30                 4.951251e+05            0.165042   
2025-09-30                 1.871500e+08            0.190000   

            Normalized EBITDA  Total Unusual Items  \
2024-09-30                NaN                  NaN   
2024-12-31       3.675500e+10        -1.129000e+09   
2025-03-31       4.032400e+10         3.870000e+08   
2025-06-30       4.443100e+10         3.000000e+06   
2025-09-30       4.707500e+10         9.850000e+08   

            Total Unusual Items Excluding Goodwill  \
2024-09-30                                     NaN   
2024-12-31                           -1.129000e+09   
2025-03-31                            3.870000e+08   
2025-06-30                            3.000000e+06   
2025-09-30                

In [14]:
fundamentals_df.dropna(inplace=True)
print(fundamentals_df.head())

Empty DataFrame
Columns: [Tax Effect Of Unusual Items, Tax Rate For Calcs, Normalized EBITDA, Total Unusual Items, Total Unusual Items Excluding Goodwill, Net Income From Continuing Operation Net Minority Interest, Reconciled Depreciation, Reconciled Cost Of Revenue, EBITDA, EBIT, Net Interest Income, Interest Expense, Interest Income, Normalized Income, Net Income From Continuing And Discontinued Operation, Total Expenses, Total Operating Income As Reported, Diluted Average Shares, Basic Average Shares, Diluted EPS, Basic EPS, Diluted NI Availto Com Stockholders, Net Income Common Stockholders, Net Income, Net Income Including Noncontrolling Interests, Net Income Continuous Operations, Tax Provision, Pretax Income, Other Income Expense, Other Non Operating Income Expenses, Special Income Charges, Write Off, Gain On Sale Of Security, Net Non Operating Interest Income Expense, Interest Expense Non Operating, Interest Income Non Operating, Operating Income, Operating Expense, Research An

In [15]:
def build_full_dataset(ticker = "MSFT", start="2020-01-01", end="2024-01-01"):
    
    # Get market sentiment once
    print("Fetching market sentiment...")
    sentiment = get_market_sentiment(start, end)
    sentiment.index = pandas.to_datetime(sentiment.index.date)
        # Get price data
    print("Fetching price data...")
    price_data = get_ticker_data(ticker, start, end)
    price_data.index = price_data.index.tz_localize(None)

    print("Fetching fundamentals...")
    fundamentals_df = get_full_historical_fundamentals(ticker)
    fundamentals_df.index = pandas.to_datetime(fundamentals_df.index.date)  # date only, no time
    fundamentals_df = fundamentals_df.sort_index()

    # Get fundamentals once per ticker
    print("Fetching fundamentals...")
    daily_index = price_data.index
    
    # Combine indexes, forward fill then backward fill
    daily_index = price_data.index
    fundamentals_daily = (
        fundamentals_df
        .reindex(fundamentals_df.index.union(daily_index))
        .sort_index()
        .ffill()
        .bfill()
        .reindex(daily_index)
    )
    print("Price data range:", price_data.index.min(), "to", price_data.index.max())
    print("Fundamentals range:", fundamentals_df.index.min(), "to", fundamentals_df.index.max())
    print(fundamentals_daily.tail())
    dataset = pandas.concat([price_data, sentiment, fundamentals_daily], axis=1)
    dataset = dataset.dropna()
    
    return dataset, sentiment, fundamentals_daily, price_data

# Build and save
dataset, sentiment, fundamentals_daily, price_data = build_full_dataset("MSFT")

print("Dataset shape:", dataset.shape)
print("Columns:", dataset.columns.tolist())


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Fetching market sentiment...
Fetching price data...


Fetching fundamentals...
Fetching fundamentals...
Price data range: 2020-01-02 00:00:00 to 2023-12-29 00:00:00
Fundamentals range: 2024-09-30 00:00:00 to 2025-12-31 00:00:00
            Tax Effect Of Unusual Items  Tax Rate For Calcs  \
Date                                                          
2023-12-22                 -203220000.0                0.18   
2023-12-26                 -203220000.0                0.18   
2023-12-27                 -203220000.0                0.18   
2023-12-28                 -203220000.0                0.18   
2023-12-29                 -203220000.0                0.18   

            Normalized EBITDA  Total Unusual Items  \
Date                                                 
2023-12-22       3.675500e+10        -1.129000e+09   
2023-12-26       3.675500e+10        -1.129000e+09   
2023-12-27       3.675500e+10        -1.129000e+09   
2023-12-28       3.675500e+10        -1.129000e+09   
2023-12-29       3.675500e+10        -1.129000e+09   

     

In [16]:
price_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'Daily_Return', 'Log_Return',
    'MA_20', 'MA_50', 'MA_200',
    'Volatility_20', 'Volume_Change', 'Volume_MA_20',
    'RSI', 'MACD', 'MACD_Signal',
    'BB_Upper', 'BB_Lower', 'BB_Width'
]
sentiment_cols = [
    'VIX', 'VIX_MA20',
    'SP500', 'SP500_Return', 'SP500_MA50',
    'Treasury_10Y', 'Yield_Change',
    'Gold_Price', 'Gold_Return',
    'DXY', 'DXY_Return',
    'Market_Regime'
]
fundamental_cols = [
    'Total Revenue', 'Gross Profit', 'Operating Income',
    'Net Income', 'EBITDA', 'EBIT',
    'Diluted EPS', 'Basic EPS',
    'Total Assets', 'Total Debt', 'Net Debt',
    'Common Stock Equity', 'Working Capital',
    'Free Cash Flow', 'Operating Cash Flow',
    'Research And Development',
    'Capital Expenditure',
    'Stock Based Compensation',
    'Diluted Average Shares', 'Ordinary Shares Number'
]
def select_relevant_columns(dataset):
    keep_cols = price_cols + sentiment_cols + fundamental_cols
    
    # Only keep columns that exist in dataset
    keep_cols = [col for col in keep_cols if col in dataset.columns]
    
    dataset = dataset[keep_cols]
    
    # Add derived fundamental ratios
    dataset['Gross_Margin'] = dataset['Gross Profit'] / dataset['Total Revenue']
    dataset['Operating_Margin'] = dataset['Operating Income'] / dataset['Total Revenue']
    dataset['Net_Margin'] = dataset['Net Income'] / dataset['Total Revenue']
    dataset['Debt_To_Equity'] = dataset['Total Debt'] / dataset['Common Stock Equity']
    dataset['ROA'] = dataset['Net Income'] / dataset['Total Assets']
    
    return dataset

dataset = select_relevant_columns(dataset)
print("Reduced shape:", dataset.shape)
print("Columns:", dataset.columns.tolist())

Reduced shape: (807, 56)
Columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'Daily_Return', 'Log_Return', 'MA_20', 'MA_50', 'MA_200', 'Volatility_20', 'Volume_Change', 'Volume_MA_20', 'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower', 'BB_Width', 'VIX', 'VIX_MA20', 'SP500', 'SP500_Return', 'SP500_MA50', 'Treasury_10Y', 'Yield_Change', 'Gold_Price', 'Gold_Return', 'DXY', 'DXY_Return', 'Market_Regime', 'Total Revenue', 'Gross Profit', 'Operating Income', 'Net Income', 'EBITDA', 'EBIT', 'Diluted EPS', 'Basic EPS', 'Total Assets', 'Total Debt', 'Net Debt', 'Common Stock Equity', 'Working Capital', 'Free Cash Flow', 'Operating Cash Flow', 'Research And Development', 'Capital Expenditure', 'Stock Based Compensation', 'Diluted Average Shares', 'Ordinary Shares Number', 'Gross_Margin', 'Operating_Margin', 'Net_Margin', 'Debt_To_Equity', 'ROA']


C:\Users\chanr\AppData\Local\Temp\ipykernel_32348\3446483507.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset['Gross_Margin'] = dataset['Gross Profit'] / dataset['Total Revenue']
C:\Users\chanr\AppData\Local\Temp\ipykernel_32348\3446483507.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset['Operating_Margin'] = dataset['Operating Income'] / dataset['Total Revenue']
C:\Users\chanr\AppData\Local\Temp\ipykernel_32348\3446483507.py:40: SettingWithCopyWarning: 
A value is trying to be set o

In [17]:
price_data[price_cols].to_csv('msft_price_data.csv', index=True)
sentiment[sentiment_cols].to_csv('msft_sentiment.csv', index=True)
fundamentals_daily[fundamental_cols].to_csv('msft_fundamentals.csv', index=True)




In [18]:
dataset.dropna(inplace=True)
dataset.drop_duplicates(inplace=True)

In [19]:
#import sys
#!{sys.executable} -m pip install tensorflow
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Dropout, Concatenate

In [20]:
daily_cols = [col for col in price_cols + sentiment_cols if col in dataset.columns]
fund_cols = [col for col in fundamental_cols if col in dataset.columns]

sequence_length = 60                          # look back 60 days
n_daily_features = len(daily_cols)            # number of daily features
n_fundamental_features = len(fund_cols)       # number of fundamental features

In [21]:
# Branch 1 - LSTM for daily time series (prices + sentiment)
daily_input = Input(shape=(sequence_length, n_daily_features))
lstm_out = LSTM(128, return_sequences=True)(daily_input)
lstm_out = Dropout(0.2)(lstm_out)
lstm_out = LSTM(64)(lstm_out)

# Branch 2 - Dense for fundamentals
fund_input = Input(shape=(n_fundamental_features,))
fund_out = Dense(32, activation='relu')(fund_input)
fund_out = Dense(16, activation='relu')(fund_out)

# Combine both branches
combined = Concatenate()([lstm_out, fund_out])
combined = Dense(32, activation='relu')(combined)
output = Dense(1)(combined)  # predicted price

model = Model(inputs=[daily_input, fund_input], outputs=output)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [22]:
from sklearn.preprocessing import MinMaxScaler
daily_cols = [col for col in price_cols + sentiment_cols if col in dataset.columns]
fund_cols = [col for col in fundamental_cols if col in dataset.columns]
daily_cols = ['Close'] + [col for col in daily_cols if col != 'Close']
print("Dataset shape before dropna:", dataset.shape)
print("NaN counts per column:")
print(dataset[daily_cols + fund_cols].isnull().sum())
dataset = dataset.dropna(subset=daily_cols + fund_cols)
missing_daily = [col for col in daily_cols if col not in dataset.columns]
missing_fund = [col for col in fund_cols if col not in dataset.columns]
print("Missing daily cols:", missing_daily)
print("Missing fund cols:", missing_fund)

# Check for non-numeric columns
print(dataset[daily_cols].dtypes)
print(dataset[fund_cols].dtypes)

# Check for infinite values
print("Infinite in daily:", np.isinf(dataset[daily_cols]).sum().sum())
print("Infinite in fund:", np.isinf(dataset[fund_cols]).sum().sum())

# Check for remaining NaN
print("NaN in daily:", dataset[daily_cols].isnull().sum().sum())
print("NaN in fund:", dataset[fund_cols].isnull().sum().sum())
daily_scaler = MinMaxScaler()
daily_scaled = daily_scaler.fit_transform(dataset[daily_cols])
fundamentals_scaler = MinMaxScaler()
fundamentals_scaled = fundamentals_scaler.fit_transform(dataset[fund_cols])


Dataset shape before dropna: (807, 56)
NaN counts per column:
Close                       0
Open                        0
High                        0
Low                         0
Volume                      0
Daily_Return                0
Log_Return                  0
MA_20                       0
MA_50                       0
MA_200                      0
Volatility_20               0
Volume_Change               0
Volume_MA_20                0
RSI                         0
MACD                        0
MACD_Signal                 0
BB_Upper                    0
BB_Lower                    0
BB_Width                    0
VIX                         0
VIX_MA20                    0
SP500                       0
SP500_Return                0
SP500_MA50                  0
Treasury_10Y                0
Yield_Change                0
Gold_Price                  0
Gold_Return                 0
DXY                         0
DXY_Return                  0
Market_Regime               0
Total Re

In [ ]:
from sklearn.model_selection import train_test_split

def create_sequences(data, sequence_length=60):
    X, y = [], []
    for i in range(sequence_length, len(data)):
        X.append(data[i-sequence_length:i])
        y.append(data[i, 0])  # predicting Close price
    return np.array(X), np.array(y)

X_daily, y = create_sequences(daily_scaled, sequence_length=60)
X_fund = fundamentals_scaled[60:]  # align with sequences
print("X_daily shape:", X_daily.shape)
print("X_fund shape:", X_fund.shape)
print("y shape:", y.shape)
split = int(len(X_daily) * 0.8)
X_daily_train = X_daily[:split]
X_daily_test  = X_daily[split:]
X_fund_train = X_fund[:split]
X_fund_test  = X_fund[split:]
y_train = y[:split]
y_test  = y[split:]
print("X_daily_train shape:", X_daily_train.head())
print("X_fund_train shape:", X_fund_train.head())
print("Train samples:", len(X_daily_train))
print("Test samples:", len(X_daily_test))

# Step 3 - Train model on training data only
model.fit(
    [X_daily_train, X_fund_train], y_train,
    epochs=50,
    batch_size=32,
    validation_data=([X_daily_test, X_fund_test], y_test)  # use test as validation
)

# Step 4 - Evaluate on test data
y_pred = model.predict([X_daily_test, X_fund_test])

# Step 5 - Inverse transform to get actual prices
y_pred_actual = daily_scaler.inverse_transform(
    np.concatenate([y_pred, np.zeros((len(y_pred), daily_scaled.shape[1]-1))], axis=1)
)[:, 0]

y_test_actual = daily_scaler.inverse_transform(
    np.concatenate([y_test.reshape(-1,1), np.zeros((len(y_test), daily_scaled.shape[1]-1))], axis=1)
)[:, 0]

# Step 6 - Check results
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
mae = mean_absolute_error(y_test_actual, y_pred_actual)

print(f"RMSE: ${rmse:.2f}")
print(f"MAE:  ${mae:.2f}")

X_daily shape: (747, 60, 31)
X_fund shape: (747, 20)
y shape: (747,)
Train samples: 597
Test samples: 150
Epoch 1/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 6s 134ms/step - loss: 0.0200 - mae: 0.1054 - val_loss: 0.0097 - val_mae: 0.0784
Epoch 2/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 108ms/step - loss: 0.0038 - mae: 0.0502 - val_loss: 0.0096 - val_mae: 0.0828
Epoch 3/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - loss: 0.0028 - mae: 0.0423 - val_loss: 0.0076 - val_mae: 0.0737
Epoch 4/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - loss: 0.0025 - mae: 0.0398 - val_loss: 0.0049 - val_mae: 0.0582
Epoch 5/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0022 - mae: 0.0373 - val_loss: 0.0031 - val_mae: 0.0472
Epoch 6/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 114ms/step - loss: 0.0019 - mae: 0.0347 - val_loss: 0.0046 - val_mae: 0.0541
Epoch 7/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - loss: 0.0023 - mae: 0.0380 - val_loss: 0.0026 - val_mae: 0.0422
Epoch 8/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - loss: 0.0020 - mae: